# N116 · 轮廓DP与窄网格状态

**目标：** 仅保存已处理区域与未来相交的边界信息。

**先修：** N112, N087, N128。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 行掩码；合法状态；相邻行兼容；多进制状态；复杂度按宽度增长。

## 从问题到算法

窄网格的指数因素应放在宽度，而非整个面积。把一行选座状态压成mask，预先排除横向相邻；相邻行只检查左上与右上冲突。三色网格用三进制行状态，先排除行内同色相邻，再检查相邻列的逐位不同。

## 最小实现

**本章接口：** `max_students_profile(seats)`、`color_the_grid(m, n)`、`num_of_ways_n3(n)`

In [1]:
from itertools import product
MOD=10**9+7

def max_students_profile(seats):
    if not seats: return 0
    width=len(seats[0]); dp={0:0}
    for row in seats:
        allowed=sum(1<<i for i,x in enumerate(row) if x=='.')
        states=[mask for mask in range(1<<width) if mask&allowed==mask and not mask&(mask<<1)]
        nxt={}
        for mask in states:
            nxt[mask]=mask.bit_count()+max(v for prev,v in dp.items() if not mask&(prev<<1) and not mask&(prev>>1))
        dp=nxt
    return max(dp.values())

def color_the_grid(m,n):
    if min(m,n)<0: raise ValueError('nonnegative dimensions required')
    if not m or not n: return 1
    states=[p for p in product(range(3),repeat=m) if all(a!=b for a,b in zip(p,p[1:]))]
    compatible=[[j for j,b in enumerate(states) if all(x!=y for x,y in zip(a,b))] for a in states]
    dp=[1]*len(states)
    for _ in range(1,n): dp=[sum(dp[j] for j in neighbors)%MOD for neighbors in compatible]
    return sum(dp)%MOD

def num_of_ways_n3(n):
    if n==0: return 1
    aba=abc=6
    for _ in range(1,n): aba,abc=(3*aba+2*abc)%MOD,(2*aba+2*abc)%MOD
    return (aba+abc)%MOD

## 正确性、前提与复杂度

给定上一行状态，未来只受该行边界影响，更早行可被遗忘。合法状态完整枚举了行内选择，兼容关系精确表达跨行约束。宽度3的三色状态按ABA与ABC两类聚合，颜色重命名对称性保证同类转移数相同，得到常数状态递推。

**代价：** 选座保守O(R·4^W)时间、O(2^W)状态空间；三色状态数S=3·2^(m−1)，预计算兼容O(mS²)，后续O(nS²)时间和O(S²)空间；宽3聚合版O(n)时间O(1)空间。指数宽度必须小。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

seats=['.#.','...']; width=3; rows=[]
for r,row in enumerate(seats):
    allowed=sum(1<<i for i,c in enumerate(row) if c=='.')
    for mask in range(1<<width):
        if mask&allowed==mask and not mask&(mask<<1): rows.append((r,format(mask,'03b'),mask.bit_count()))
show_table(['行','合法座位掩码','人数'],rows)
print('全局最多人数',max_students_profile(seats))

行,合法座位掩码,人数
0,000,0
0,001,1
0,100,1
0,101,2
1,000,0
1,001,1
1,010,1
1,100,1
1,101,2


全局最多人数 4


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert color_the_grid(1,1)==3 and num_of_ways_n3(1)==12
assert max_students_profile(['...'])==2
for n in range(1,9): assert num_of_ways_n3(n)==color_the_grid(3,n)
for broken in range(1<<6):
    seats=[''.join('#' if broken>>(r*3+c)&1 else '.' for c in range(3)) for r in range(2)]
    valid=[]
    for mask in range(1<<6):
        if mask&broken: continue
        cells=[(i//3,i%3) for i in range(6) if mask>>i&1]
        if all(not(abs(c-d)==1 and abs(r-s)<=1) for i,(r,c) in enumerate(cells) for s,d in cells[i+1:]): valid.append(mask.bit_count())
    assert max_students_profile(seats)==max(valid)
for m,n in [(1,3),(2,2),(2,3)]:
    count=0
    for colors in product(range(3),repeat=m*n):
        if all(colors[r*n+c]!=colors[(r+1)*n+c] for r in range(m-1) for c in range(n)) and all(colors[r*n+c]!=colors[r*n+c+1] for r in range(m) for c in range(n-1)): count+=1
    assert color_the_grid(m,n)==count
print('N116: 所有本章断言通过')

N116: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 交换长宽以减少状态数但核对规则对称性。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 与二分匹配解考场题对照。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1349. Maximum Students Taking Exam（canonical）：[官方入口](https://leetcode.com/problems/maximum-students-taking-exam/)
- 1931. Painting a Grid With Three Different Colors（canonical）
- 1411. Number of Ways to Paint N × 3 Grid（canonical）：[官方入口](https://leetcode.com/problems/number-of-ways-to-paint-n-3-grid/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。